# Building the Data Foundation for Casper's Ghost Kitchen

## Meet Casper: Running a Modern Ghost Kitchen Empire

Casper operates a cutting-edge **ghost kitchen** network - commercial cooking facilities designed exclusively for delivery orders. With no dining rooms, just optimized kitchen space, Casper can operate multiple restaurant brands from single locations across major cities.

**The Challenge:** Every day, Casper processes hundreds of orders across multiple virtual restaurant brands like "Mediterranean Express," "Burger Boulevard," and "Asian Fusion Co." Each order generates a complex stream of real-time events:

- **Order Creation** - Customer places order with multiple items
- **Kitchen Events** - Order confirmed, prep started, cooking, ready for pickup  
- **Driver Events** - Driver assigned, arrived at kitchen, picked up order
- **Real-time Tracking** - GPS pings every minute showing delivery progress
- **Final Delivery** - Order delivered with confirmation coordinates

**The Business Problem:** This creates a massive stream of operational events - thousands per day - all arriving as separate JSON payloads. Without proper data architecture, Casper's team faces:
- No unified view of order performance across locations
- Cannot identify delivery bottlenecks or late orders  
- Unable to optimize kitchen operations or driver routing
- No data foundation for AI-powered business decisions

## The Solution: A Real-Time Data Pipeline with Medallion Architecture

In this notebook, we'll build Casper's data foundation using **Databricks Lakeflow** - a modern approach to real-time data processing that transforms raw operational events into business-ready insights.

We'll implement two critical data pipelines:

### **Bronze → Silver → Gold Pipeline (`dlt_order_items`)**
Transforms raw JSON events into clean, analyzable data using the medallion architecture:
- **Bronze Layer**: Land raw JSON events exactly as they arrive from Casper's operations
- **Silver Layer**: Clean and normalize data - explode order items, validate timestamps, compute line totals  
- **Gold Layer**: Create business-ready aggregated tables for analytics, dashboards, and AI applications

### **Orders In Progress Stream**
Real-time operational intelligence:
- Tracks every order from creation through delivery
- Maintains live state of in-flight orders for operational dashboards
- Enables real-time alerts for delayed or problem orders

**Why This Matters for Casper's Business:**
- **Operational Excellence**: Monitor delivery times, kitchen performance, and driver efficiency in real-time
- **Revenue Optimization**: Identify high-performing menu items, peak hours, and profitable locations
- **Customer Experience**: Detect and resolve delivery issues before customers complain
- **AI-Ready Foundation**: Clean, structured data powers intelligent recommendations and automated decisions

Let's build Casper's data foundation that turns operational chaos into competitive advantage!

## Understanding Casper's Data Before We Begin

Before we build the pipeline, let's understand what data Casper's operations actually generate. Each order creates multiple timestamped events throughout its lifecycle:

**Sample Order Journey:**
1. **Order Created**: Customer orders 2x Falafel Bowls and 1x Harissa Chips from "Mediterranean Express" 
2. **Kitchen Progression**: Order confirmed → prep started → cooking → ready for pickup
3. **Driver Assignment**: Driver receives order, arrives at kitchen, picks up food
4. **Real-time Delivery**: GPS coordinates transmitted every minute during delivery
5. **Delivery Confirmed**: Final location coordinates when order reaches customer

This generates 10-15 separate JSON events per order. With hundreds of daily orders across multiple locations, Casper needs a robust data pipeline to handle this complexity.

### Setup

We need to initialize data and catalogs before we begin creating our pipeline.

In [0]:
from utils.utils import (
    setup_catalog_and_volume,
    copy_raw_data_to_volume,
    drop_gk_demo_catalog,
    initialize_dimension_tables
)

# Drop existing catalog/volume/table if you need to start fresh
drop_gk_demo_catalog(spark)

## 1. Setup the catalog and volume
setup_catalog_and_volume(spark)

## 2. Copy the raw data to the volume
copy_raw_data_to_volume()

## 3. Initialize the static dimension tables
initialize_dimension_tables(spark)

## Building Casper's Order Processing Pipeline

Now we'll create the `order_items_dlt` pipeline that transforms Casper's raw operational events into business-ready data tables.

**Why Casper Needs This Pipeline:**
- **Volume**: Hundreds of orders daily = thousands of individual JSON events
- **Variety**: Different event types (orders, kitchen updates, GPS pings, deliveries) with different schemas
- **Velocity**: Real-time events arriving continuously throughout business hours
- **Business Value**: Operations teams need clean, queryable data to optimize performance

#### `order_items_dlt` Declarative Pipeline

The code for the pipeline is prepared in the `./pipelines/order_items_dlt` directory. To initialize this code as a declarative pipeline, we need to go to `Jobs & Pipelines` in the main navigation bar and click `Create` and then select `ETL Pipeline`

![](./images/lakeflow/1.png)


In the new page you'll need to:

1. Name the pipeline in the top left corner (**order_items**)
2. Select `gk_demo` catalog and create a new schema `lakeflow` for all the pipeline assets 
3. Select `Add existing assets` and select the folder `./pipelines/order_items_dlt/` in this repository for both paths


![](./images/lakeflow/2.png)
![](./images/lakeflow/3.png)


Once you add the pipeline code you'll see this page that provides:

1. All pipeline assets (code) in the left hand pane
2. Tab based editor in the center pane
3. Table & Performance results in the bottom pane 
4. A visual dependency graph in the right hand pane

![](./images/lakeflow/4.png)


Click `Run Pipeline` to start the pipeline and watch the panes populate with the results

![](./images/lakeflow/5.png)

## Understanding What We've Built: Casper's Data Architecture

We've now completed the main declarative pipeline for Casper's operations. This medallion architecture transforms raw operational events into actionable business intelligence tables. As such, using the Databricks Assistant in the Catalog Explorer, you can query the managed tables in natural language for actionable insights.

### **Bronze Layer - Raw Event Capture**
**Business Value**: Complete audit trail of all operational events
- Ingests raw JSON events exactly as they arrive from Casper's systems
- Preserves original data for compliance and debugging
- Uses Auto Loader to automatically process new events as they arrive

### **Silver Layer - Clean Operational Data** 
**Business Value**: Queryable, normalized order data for analysis
- Focuses on `order_created` events (the orders themselves)
- Explodes the items array so each menu item becomes a separate row
- Calculates `extended_price` (price × quantity) for revenue analysis
- Converts string timestamps to proper datetime types for time-series analysis

**Real-World Impact**: Operations teams can now use Databricks Assistant to query "Show me all orders from San Francisco location containing falafel items yesterday" - something impossible with raw JSON events.

### **Gold Layer - Business Intelligence Tables**

#### **Order Header Table**: Complete Order Analysis
- One row per order with total revenue, item count, and brand mix
- **Use Case**: "Which orders have the highest average value?" "How many multi-brand orders do we get?"

#### **Daily Item Sales**: Menu Performance Intelligence  
- Daily sales performance for every menu item across all brands
- **Use Case**: "Which items should we promote?" "What's our best-selling Mediterranean dish?"

#### **Daily Brand Sales**: Brand Portfolio Analysis
- Revenue and order volume by brand (Mediterranean Express, Burger Boulevard, etc.)
- Uses HyperLogLog for efficient distinct order counting in streaming data
- **Use Case**: "Which virtual restaurant brands are most profitable?" "Should we expand Asian Fusion Co.?"

#### **Hourly Location Sales**: Operational Optimization
- Hour-by-hour performance metrics for each ghost kitchen location
- **Use Case**: "When are our San Francisco peak hours?" "Which locations need more driver coverage at 7 PM?"

### **Streaming Architecture Benefits**
- **3-Hour Watermark**: Handles late-arriving data (orders that take time to process) while keeping system performant
- **Approximate Counting**: Uses probabilistic algorithms for scalable distinct counts without memory issues  
- **Smart Partitioning**: Data organized by date/hour for lightning-fast time-based queries

### **The Business Transformation**
Before this pipeline, Casper's team had thousands of scattered JSON files. Now they have:
- **Real-time visibility** into kitchen and delivery performance
- **Data-driven insights** for menu optimization and location expansion  
- **Foundation for AI** applications like intelligent routing and demand forecasting
- **Operational efficiency** through automated monitoring and alerting

By default, everything runs in manual trigger mode to save resources, but you can change it to continuous streaming mode for real-time processing.

## Real-Time Operational Intelligence: Orders In Progress Stream

While our medallion pipeline handles historical analysis perfectly, Casper's operations team needs something more: **real-time visibility into active orders**.

**The Operational Challenge:**
- Kitchen managers need to see which orders are currently being prepared
- Drivers need to know which orders are ready for pickup
- Customer service needs to track delayed orders before customers complain
- Operations managers need alerts when orders are taking too long

**The Solution:** A streaming job that maintains live state of every order from creation through delivery.

### `Orders In Progress` streaming job

The `Orders In Progress` streaming job aggregates all events for each order in real-time and maintains them in streaming state until the order completes. This gives Casper's team a live operational dashboard.

**How it works:**
1. Reads from the Bronze `all_events` table as events arrive
2. Groups events by `order_id` and maintains them in Spark's streaming state  
3. Tracks the complete journey: order_created → kitchen_confirmed → prep_started → cooking → ready_for_pickup → driver_assigned → picked_up → en_route → delivered
4. Keeps orders "in progress" until the final `delivered` event arrives
5. Provides live query capability for operational dashboards

The code for this streaming job is defined using a notebook and is located at `./notebooks/Orders In Progress.ipynb`.

Starting the `Jobs & Pipelines` tab in the main left navigation bar, click `Create > Job`

![](./images/lakeflow/job1.png)


Give the job a new title (**Orders In Progress Stream**) and click `Notebook` under `Add your first task`

![](./images/lakeflow/job2.png)


Give your task a name (`main`) and specify the source of the job as the notebook path as shown in the image. Finally select `Create Task` and `Run Now` to start the first run of this streaming job.

**Note:** the streaming job is written using the `AvailableNow` trigger, so it will execute once and stop. This is done to save resources in the Free edition. To process new data (in future examples), you'll need to manually run the job again.

![](./images/lakeflow/job3.png)

## Operational Intelligence in Action

Once the job finishes running, we can query the live operational state - this is the real-time data that powers Casper's operational dashboards and alerts.

**Business Value of Live State:**
- **Kitchen Operations**: "Show me all orders currently in prep or cooking phase"  
- **Driver Management**: "Which orders are ready for pickup but don't have drivers assigned?"
- **Customer Service**: "This customer called about their order - where is it in the process?"
- **Performance Monitoring**: "Alert me when any order has been in cooking phase for more than 20 minutes"

The query below leverages [Spark Streaming's State Reader API](https://docs.databricks.com/aws/en/structured-streaming/read-state) to access live operational state.

This same query powers the operational dashboards in Casper's management systems and will be used in the `Apps` demo to surface orders in progress to a Databricks App.

(**Note** blocked on ES-1538976)

In [0]:
%sql

SELECT 
    key.order_id as order_id,
    collect_list(list_element) as events 
FROM read_statestore(
    "/Volumes/gk_demo/default/checkpoints/orders_in_progress",
    stateVarName => 'events' ) 
GROUP BY key.order_id